# Phase C2 — Comprehensive Behavioural, Relational, Statistical and ML EDA

This notebook analyses the **actual normalized Dataset V1**.

Important:
- Raw files are not modified.
- Latent sidecar data is not used.
- Findings are calculated from the observed data.
- Simulator assumptions are not treated as empirical findings.
- No model training, splitting, or final feature freezing occurs in C2.

The purpose is to understand the complete recovery environment and produce evidence for C3 feature engineering and final feature selection.

In [14]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CURRENT = Path.cwd()

if (CURRENT / "data" / "normalized").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data" / "normalized").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise FileNotFoundError(
        "Could not locate project root containing data/normalized."
    )

DATA_DIR = PROJECT_ROOT / "data" / "normalized"
CONTRACT_DIR = PROJECT_ROOT / "reports" / "ml_contract"
QUALITY_DIR = PROJECT_ROOT / "reports" / "data_quality"
REPORT_DIR = PROJECT_ROOT / "reports" / "eda"
DOCS_DIR = PROJECT_ROOT / "docs"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

TABLES = [
    "customers",
    "subscriptions",
    "payment_attempts",
    "recovery_cases",
    "recovery_events",
    "action_log",
    "case_outcomes",
    "ml_datamart",
]

data = {}
for name in TABLES:
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing normalized table: {path}")
    data[name] = pd.read_csv(path, low_memory=False)

print("Loaded normalized tables:")
for name, df in data.items():
    print(f"{name:20s} {len(df):>10,} rows x {len(df.columns)} cols")

ml = data["ml_datamart"]

if "y" in ml.columns:
    target_col = "y"
elif "y_intervention" in ml.columns:
    target_col = "y_intervention"
else:
    raise ValueError("No target column y or y_intervention found.")

ml[target_col] = pd.to_numeric(ml[target_col], errors="coerce")


Loaded normalized tables:
customers                20,000 rows x 5 cols
subscriptions            22,078 rows x 12 cols
payment_attempts        264,936 rows x 11 cols
recovery_cases          103,026 rows x 19 cols
recovery_events         986,365 rows x 9 cols
action_log              157,423 rows x 20 cols
case_outcomes           103,026 rows x 17 cols
ml_datamart             157,423 rows x 45 cols


## 1. Dataset Scope and Quality Context

In [15]:

rows = []

for name, df in data.items():
    total_cells = df.shape[0] * df.shape[1]
    rows.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_pct": (
            100 * df.isna().sum().sum() / total_cells
            if total_cells else 0.0
        ),
        "duplicate_rows": int(df.duplicated().sum()),
    })

table_summary = pd.DataFrame(rows)
display(table_summary)

table_summary.to_csv(
    REPORT_DIR / "table_eda_summary.csv",
    index=False,
)


,table,rows,columns,missing_cells,missing_pct,duplicate_rows
0,customers,20000,5,0,0.000000,0
1,subscriptions,22078,12,0,0.000000,0
2,payment_attempts,264936,11,238138,8.171373,0
3,recovery_cases,103026,19,68826,3.516026,0
4,recovery_events,986365,9,0,0.000000,0
5,action_log,157423,20,203028,6.448486,0
6,case_outcomes,103026,17,68826,3.929676,0
7,ml_datamart,157423,45,203028,2.865994,0


## 2. Exact ML Grain and Repeated-Entity Structure

In [16]:

grain = []

for col in [
    "action_id",
    "case_id",
    "customer_id",
    "subscription_id",
    "logging_policy",
]:
    if col in ml.columns:
        grain.append({
            "metric": f"unique_{col}",
            "value": ml[col].nunique(dropna=True),
        })

grain.append({
    "metric": "total_rows",
    "value": len(ml),
})

if "case_id" in ml.columns:
    rows_per_case = ml.groupby("case_id").size()
    grain.append({
        "metric": "mean_rows_per_case",
        "value": rows_per_case.mean(),
    })
    grain.append({
        "metric": "median_rows_per_case",
        "value": rows_per_case.median(),
    })

display(pd.DataFrame(grain))

if {"case_id", "decision_seq"}.issubset(ml.columns):
    duplicate_case_decision = int(
        ml.duplicated(
            subset=["case_id", "decision_seq"]
        ).sum()
    )
    print(
        "Duplicate (case_id, decision_seq) rows:",
        duplicate_case_decision
    )

if "action_id" in ml.columns:
    print(
        "Duplicate action_id rows:",
        int(ml["action_id"].duplicated().sum())
    )


,metric,value
0,unique_case_id,25775.000000
1,unique_customer_id,10924.000000
2,unique_subscription_id,11596.000000
3,unique_logging_policy,3.000000
4,total_rows,157423.000000
5,mean_rows_per_case,6.107585
6,median_rows_per_case,5.000000


Duplicate (case_id, decision_seq) rows: 74061


## 3. Customers — Distribution and Customer Structure

In [17]:

cust = data["customers"]

display(cust.describe(include="all").T)

for col in ["age_band", "state", "locale_pref"]:
    if col in cust.columns:
        summary = (
            cust[col]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )
        summary["proportion"] = (
            summary["count"] / len(cust)
        )
        display(summary)

if "customer_id" in data["subscriptions"].columns:
    subscriptions_per_customer = (
        data["subscriptions"]
        .groupby("customer_id")
        .size()
    )
    display(
        subscriptions_per_customer
        .describe()
        .to_frame("subscriptions_per_customer")
    )


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,20000.0,NaN,NaN,NaN,10000.5,5773.647028,1.0,5000.75,10000.5,15000.25,20000.0
age_band,20000,5,AGE_26_35,6935,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,20000,10,MAHARASHTRA,3009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
locale_pref,20000,3,EN,10914,NaN,NaN,NaN,NaN,NaN,NaN,NaN
signed_up_date,20000,730,2022-04-22 00:00:00,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count,proportion
age_band,,
AGE_26_35,6935,0.34675
AGE_36_45,5050,0.25250
AGE_18_25,4977,0.24885
AGE_46_55,2019,0.10095
AGE_56_PLUS,1019,0.05095


,count,proportion
state,,
MAHARASHTRA,3009,0.15045
UTTAR_PRADESH,2392,0.11960
KARNATAKA,2353,0.11765
DELHI,2064,0.10320
TAMIL_NADU,1925,0.09625
TELANGANA,1803,0.09015
ANDHRA_PRADESH,1646,0.08230
GUJARAT,1623,0.08115
RAJASTHAN,1605,0.08025


,count,proportion
locale_pref,,
EN,10914,0.54570
HI,5079,0.25395
HI_EN,4007,0.20035


,subscriptions_per_customer
count,20000.000000
mean,1.103900
std,0.305138
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,2.000000


## 4. Subscriptions — Commercial and Payment Setup

In [18]:

subs = data["subscriptions"]

for col in [
    "segment",
    "plan_tier",
    "cadence",
    "rail",
    "bank_name",
    "status",
]:
    if col in subs.columns:
        display(
            subs[col]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

for col in [
    "plan_amount_inr",
    "billing_day_of_month",
    "mandate_max_amount_inr",
]:
    if col in subs.columns:
        display(
            subs[col]
            .describe()
            .to_frame()
        )


,count
segment,
OTT_STREAMING,7730
NEWS_MEDIA,4408
FITNESS,3350
EDUCATION,3329
SAAS_SMB,3261


,count
plan_tier,
BASIC,12297
STANDARD,6533
PREMIUM,3248


,count
cadence,
MONTHLY,22078


,count
rail,
UPI_AUTOPAY,15579
CARD_SI,6499


,count
bank_name,
SBI,5494
HDFC,3337
ICICI,2635
AXIS,2248
PNB,2231
INDUSIND,2180
YES_BANK,2143
KOTAK,1810


,count
status,
ACTIVE,22078


,plan_amount_inr
count,22078.000000
mean,512.194130
std,696.210813
min,49.000000
25%,149.000000
50%,299.000000
75%,599.000000
max,3999.000000


,billing_day_of_month
count,22078.000000
mean,14.473005
std,8.073011
min,1.000000
25%,7.000000
50%,15.000000
75%,21.000000
max,28.000000


,mandate_max_amount_inr
count,22078.000000
mean,771.589818
std,1043.486426
min,80.000000
25%,230.000000
50%,450.000000
75%,900.000000
max,6000.000000


## 5. Payment Attempts — Failure, Attempt and Recovery Behaviour

In [19]:

attempts = data["payment_attempts"]

for col in [
    "outcome",
    "failure_class",
    "is_recovery_attempt",
]:
    if col in attempts.columns:
        display(
            attempts[col]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

for col in ["attempt_index", "amount_inr"]:
    if col in attempts.columns:
        display(
            attempts[col]
            .describe()
            .to_frame()
        )

if "attempt_index" in attempts.columns:
    display(
        attempts["attempt_index"]
        .value_counts()
        .sort_index()
        .rename("count")
        .to_frame()
    )


,count
outcome,
SUCCESS,238138
FAILED,26798


,count
failure_class,
NaN,238138
INSUFFICIENT_FUNDS,13270
BANK_TECHNICAL,5127
MANDATE_PAUSED,3968
MANDATE_CLOSED,3647
CARD_EXPIRED,452
CARD_LOST_STOLEN,334


,count
is_recovery_attempt,
False,264936


,attempt_index
count,264936.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


,amount_inr
count,264936.00000
mean,512.19413
std,696.19636
min,49.00000
25%,149.00000
50%,299.00000
75%,599.00000
max,3999.00000


,count
attempt_index,
1,264936


## 6. Recovery Cases — Case-Level Behaviour

In [20]:

cases = data["recovery_cases"]

for col in [
    "policy_id",
    "first_failure_class",
    "current_failure_class",
    "value_tier",
    "resolution",
]:
    if col in cases.columns:
        display(
            cases[col]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

for col in [
    "amount_at_risk",
    "n_decisions",
    "retries_used",
    "contacts_sent",
    "cost_total_inr",
    "fatigue_total",
    "recovered_amount_inr",
    "time_to_recovery_h",
]:
    if col in cases.columns:
        display(
            cases[col]
            .describe()
            .to_frame()
        )

if {"policy_id", "case_id"}.issubset(cases.columns):
    policy_case_counts = (
        cases.groupby("policy_id")
        .size()
        .rename("policy_case_rows")
        .to_frame()
    )
    display(policy_case_counts)


,count
policy_id,
B0_NONE,26798
B1_PSP_PROXY,25625
B3_RANDOM_LEGAL,25332
B2_DOCTRINE,25271


,count
first_failure_class,
INSUFFICIENT_FUNDS,53353
BANK_TECHNICAL,20623
MANDATE_CLOSED,14588
MANDATE_PAUSED,12814
CARD_EXPIRED,1034
CARD_LOST_STOLEN,614


,count
current_failure_class,
INSUFFICIENT_FUNDS,54314
BANK_TECHNICAL,20019
MANDATE_CLOSED,14588
MANDATE_PAUSED,11176
RATE_LIMIT,1547
CARD_EXPIRED,853
CARD_LOST_STOLEN,529


,count
value_tier,
LOW,62020
MID,30799
HIGH,10207


,count
resolution,
EXPIRED_WINDOW,61089
RECOVERED,34200
STOPPED_POLICY,5162
ESCALATED_UNRESOLVED,1533
STOPPED_OPT_OUT,1042


,n_decisions
count,103026.000000
mean,5.509289
std,5.586530
min,0.000000
25%,2.000000
50%,2.000000
75%,9.000000
max,24.000000


... (3 outputs left out)

,fatigue_total
count,103026.000000
mean,0.565731
std,0.855803
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,4.000000


,recovered_amount_inr
count,103026.000000
mean,204.846835
std,552.912071
min,0.000000
25%,0.000000
50%,0.000000
75%,149.000000
max,3999.000000


,time_to_recovery_h
count,34200.000000
mean,61.892807
std,72.086112
min,3.000000
25%,18.000000
50%,24.000000
75%,71.000000
max,385.000000


,policy_case_rows
policy_id,
B0_NONE,26798
B1_PSP_PROXY,25625
B2_DOCTRINE,25271
B3_RANDOM_LEGAL,25332


## 7. Recovery Events — Lifecycle and Temporal Behaviour

In [21]:

events = data["recovery_events"]

if "event_type" in events.columns:
    display(
        events["event_type"]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

if "policy_id" in events.columns:
    display(
        events["policy_id"]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

if {"case_id", "ts_h"}.issubset(events.columns):
    ordered = events.sort_values(["case_id", "ts_h"])

    non_monotonic = (
        ordered.groupby("case_id")["ts_h"]
        .apply(lambda s: not s.is_monotonic_increasing)
        .sum()
    )

    print(
        "Cases with non-monotonic ts_h:",
        int(non_monotonic)
    )

    event_counts = (
        events.groupby("case_id")
        .size()
        .describe()
    )

    display(
        event_counts.to_frame("events_per_case")
    )


,count
event_type,
DECISION_MADE,574486
CASE_OPENED,103026
CASE_CLOSED,103026
ACTION_EXECUTED,101514
NOTIFICATION_SENT,55909
PAYMENT_SUCCEEDED,34200
CUSTOMER_RESPONDED,8852
HUMAN_ACTION,1724
ESCALATION_CREATED,1724


,count
policy_id,
B1_PSP_PROXY,380458
B3_RANDOM_LEGAL,296350
B2_DOCTRINE,202365
B0_NONE,107192


Cases with non-monotonic ts_h: 0


,events_per_case
count,26798.000000
mean,36.807411
std,14.667113
min,4.000000
25%,22.000000
50%,40.000000
75%,49.000000
max,72.000000


## 8. Action Log — Intervention and Policy Behaviour

In [22]:

actions = data["action_log"]

for col in [
    "action_type",
    "channel",
    "policy_id",
    "status",
]:
    if col in actions.columns:
        display(
            actions[col]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

if {"policy_id", "action_type"}.issubset(actions.columns):
    policy_action_counts = pd.crosstab(
        actions["policy_id"],
        actions["action_type"],
    )

    policy_action_share = pd.crosstab(
        actions["policy_id"],
        actions["action_type"],
        normalize="index",
    )

    print("Policy × action counts:")
    display(policy_action_counts)

    print("Policy × action proportions:")
    display(policy_action_share.round(4))

    policy_action_counts.to_csv(
        REPORT_DIR / "policy_action_counts.csv"
    )
    policy_action_share.to_csv(
        REPORT_DIR / "policy_action_share.csv"
    )

if "candidate_window_days_offset" in actions.columns:
    display(
        actions["candidate_window_days_offset"]
        .value_counts(dropna=False)
        .sort_index()
        .rename("count")
        .to_frame()
    )

if "case_id" in actions.columns:
    actions_per_case = actions.groupby("case_id").size()
    display(
        actions_per_case.describe()
        .to_frame("actions_per_case")
    )


,count
action_type,
RETRY,101514
SEND_PAYMENT_LINK,30884
SEND_REMINDER,22649
REQUEST_PM_UPDATE,2376


,count
channel,
NaN,101514
SMS,31410
WHATSAPP,12285
EMAIL,12214


,count
policy_id,
B3_RANDOM_LEGAL,64926
B1_PSP_PROXY,49915
B2_DOCTRINE,42582


Policy × action counts:


action_type,REQUEST_PM_UPDATE,RETRY,SEND_PAYMENT_LINK,SEND_REMINDER
policy_id,,,,
B1_PSP_PROXY,0,42791,7124,0
B2_DOCTRINE,259,30849,6314,5160
B3_RANDOM_LEGAL,2117,27874,17446,17489


Policy × action proportions:


action_type,REQUEST_PM_UPDATE,RETRY,SEND_PAYMENT_LINK,SEND_REMINDER
policy_id,,,,
B1_PSP_PROXY,0.0000,0.8573,0.1427,0.0000
B2_DOCTRINE,0.0061,0.7245,0.1483,0.1212
B3_RANDOM_LEGAL,0.0326,0.4293,0.2687,0.2694


,actions_per_case
count,25775.000000
mean,6.107585
std,2.739106
min,1.000000
25%,4.000000
50%,5.000000
75%,8.000000
max,13.000000


## 9. Case Outcomes — Terminal Behaviour

In [23]:

outcomes = data["case_outcomes"]

for col in [
    "resolution",
    "recovered_flag",
    "policy_id",
]:
    if col in outcomes.columns:
        display(
            outcomes[col]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

for col in [
    "recovered_amount_inr",
    "recovery_rate",
    "attempts_used",
    "contacts_used",
    "cost_total_inr",
    "fatigue_total",
    "time_to_recovery_h",
]:
    if col in outcomes.columns:
        display(
            outcomes[col]
            .describe()
            .to_frame()
        )


,count
resolution,
EXPIRED_WINDOW,61089
RECOVERED,34200
STOPPED_POLICY,5162
ESCALATED_UNRESOLVED,1533
STOPPED_OPT_OUT,1042


,count
recovered_flag,
False,68826
True,34200


,count
policy_id,
B0_NONE,26798
B1_PSP_PROXY,25625
B3_RANDOM_LEGAL,25332
B2_DOCTRINE,25271


,recovered_amount_inr
count,103026.000000
mean,204.846835
std,552.912071
min,0.000000
25%,0.000000
50%,0.000000
75%,149.000000
max,3999.000000


,recovery_rate
count,103026.000000
mean,0.331955
std,0.470917
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


,cost_total_inr
count,103026.000000
mean,1.464603
std,6.475767
min,0.000000
25%,0.000000
50%,0.500000
75%,1.150000
max,51.750000


,fatigue_total
count,103026.000000
mean,0.565731
std,0.855803
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,4.000000


,time_to_recovery_h
count,34200.000000
mean,61.892807
std,72.086112
min,3.000000
25%,18.000000
50%,24.000000
75%,71.000000
max,385.000000


## 10. ML Target and Observed Intervention Outcomes

In [24]:

print("Target:", target_col)
print("Missing target:", int(ml[target_col].isna().sum()))
print("Observed positive rate:", float(ml[target_col].mean()))

dimensions = [
    "action_type",
    "logging_policy",
    "failure_class",
    "failure_class_at_decision",
    "rail",
    "amount_bucket",
    "value_tier",
]

target_tables = []

for dim in dimensions:
    if dim not in ml.columns:
        continue

    result = (
        ml.groupby(dim)[target_col]
        .agg(
            observations="size",
            observed_success_rate="mean",
        )
        .reset_index()
        .sort_values(
            "observed_success_rate",
            ascending=False,
        )
    )

    result["dimension"] = dim
    target_tables.append(result)

    print(f"Observed success by {dim}:")
    display(result)

if target_tables:
    pd.concat(
        target_tables,
        ignore_index=True,
    ).to_csv(
        REPORT_DIR / "target_by_dimension.csv",
        index=False,
    )


Target: y
Missing target: 0
Observed positive rate: 0.2281305781239082
Observed success by action_type:


,action_type,observations,observed_success_rate,dimension
1,RETRY,101514,0.266574,action_type
0,REQUEST_PM_UPDATE,2376,0.242003,action_type
2,SEND_PAYMENT_LINK,30884,0.176499,action_type
3,SEND_REMINDER,22649,0.124774,action_type


Observed success by logging_policy:


,logging_policy,observations,observed_success_rate,dimension
1,B2_DOCTRINE,42582,0.304871,logging_policy
2,B3_RANDOM_LEGAL,64926,0.205804,logging_policy
0,B1_PSP_PROXY,49915,0.191706,logging_policy


Observed success by failure_class_at_decision:


,failure_class_at_decision,observations,observed_success_rate,dimension
0,BANK_TECHNICAL,18475,0.772395,failure_class_at_decision
6,RATE_LIMIT,2719,0.370357,failure_class_at_decision
4,MANDATE_CLOSED,15582,0.265563,failure_class_at_decision
1,CARD_EXPIRED,824,0.219660,failure_class_at_decision
2,CARD_LOST_STOLEN,400,0.212500,failure_class_at_decision
3,INSUFFICIENT_FUNDS,106159,0.137473,failure_class_at_decision
5,MANDATE_PAUSED,13264,0.123492,failure_class_at_decision


Observed success by rail:


,rail,observations,observed_success_rate,dimension
0,CARD_SI,33098,0.284609,rail
1,UPI_AUTOPAY,124325,0.213095,rail


Observed success by value_tier:


,value_tier,observations,observed_success_rate,dimension
1,LOW,84947,0.294454,value_tier
2,MID,53031,0.164225,value_tier
0,HIGH,19445,0.112677,value_tier


## 11. Candidate Feature Coverage and Distribution

In [25]:

candidate_path = (
    CONTRACT_DIR / "candidate_features.csv"
)

if not candidate_path.exists():
    raise FileNotFoundError(
        f"Missing C0 candidate feature contract: {candidate_path}"
    )

candidates = pd.read_csv(candidate_path)

feature_rows = []

for col in candidates["column"].dropna().unique():

    if col not in ml.columns:
        continue

    s = ml[col]

    row = {
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": float(100 * s.isna().mean()),
        "nunique": int(s.nunique(dropna=True)),
    }

    # ------------------------------------------------------------
    # Numeric features EXCLUDING boolean features
    # ------------------------------------------------------------
    is_numeric = (
        pd.api.types.is_numeric_dtype(s)
        and not pd.api.types.is_bool_dtype(s)
    )

    if is_numeric:

        numeric = pd.to_numeric(
            s,
            errors="coerce"
        ).dropna()

        if len(numeric) > 0:

            row.update({
                "mean": float(numeric.mean()),
                "median": float(numeric.median()),
                "std": float(numeric.std()),
                "min": float(numeric.min()),
                "max": float(numeric.max()),
                "q01": float(numeric.quantile(0.01)),
                "q99": float(numeric.quantile(0.99)),
                "zero_pct": float(
                    100 * (pd.to_numeric(s, errors="coerce") == 0).mean()
                ),
                "skew": float(numeric.skew()),
            })

    # ------------------------------------------------------------
    # Boolean features
    # ------------------------------------------------------------
    elif pd.api.types.is_bool_dtype(s):

        row.update({
            "true_pct": float(
                100 * s.fillna(False).mean()
            ),
            "false_pct": float(
                100 * (~s.fillna(False)).mean()
            ),
        })

    # ------------------------------------------------------------
    # Categorical / string features
    # ------------------------------------------------------------
    else:

        vc = s.value_counts(
            dropna=False
        )

        row.update({
            "top_category": (
                vc.index[0]
                if len(vc) > 0
                else None
            ),
            "top_category_pct": (
                float(100 * vc.iloc[0] / len(s))
                if len(vc) > 0
                else np.nan
            ),
        })

    feature_rows.append(row)

feature_summary = pd.DataFrame(feature_rows)

display(feature_summary)

feature_summary.to_csv(
    REPORT_DIR / "feature_eda_summary.csv",
    index=False
)

,feature,dtype,missing_pct,nunique,top_category,top_category_pct,mean,median,std,min,max,q01,q99,zero_pct,skew,true_pct,false_pct
0,rail,object,0.000000,2,UPI_AUTOPAY,78.975118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,bank_name,object,0.000000,8,SBI,25.848192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,segment,object,0.000000,5,OTT_STREAMING,25.809443,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,plan_tier,object,0.000000,3,BASIC,41.143289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,amount_inr,int64,0.000000,13,NaN,NaN,886.807055,499.000000,1030.924774,49.0,3999.000000,49.0,3999.000000,0.000000,1.992546,NaN,NaN
5,value_tier,object,0.000000,3,LOW,53.960984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,subscription_age_days,int64,0.000000,1033,NaN,NaN,371.390737,348.000000,192.239135,4.0,1049.000000,40.0,882.000000,0.000000,0.594843,NaN,NaN
7,sub_billing_day,int64,0.000000,28,NaN,NaN,15.249544,16.000000,8.584913,1.0,28.000000,1.0,28.000000,0.000000,-0.139123,NaN,NaN
8,decision_seq,int64,0.000000,16,NaN,NaN,1.272654,1.000000,1.715670,0.0,15.000000,0.0,8.000000,42.120910,2.195121,NaN,NaN
9,failure_class_at_decision,object,0.000000,7,INSUFFICIENT_FUNDS,67.435508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 12. Feature × Target Relationships

In [26]:

relationship_rows = []

for col in candidates["column"].dropna().unique():

    if col not in ml.columns or col == target_col:
        continue

    s = ml[col]

    # Categorical / low-cardinality analysis
    if (
        not pd.api.types.is_numeric_dtype(s)
        or s.nunique(dropna=True) <= 15
    ):

        grouped = (
            ml.groupby(col, dropna=False)[target_col]
            .agg(
                observations="size",
                observed_success_rate="mean",
            )
            .reset_index()
        )

        grouped = grouped[
            grouped["observations"] > 0
        ]

        if len(grouped) >= 2:
            diff = (
                grouped["observed_success_rate"].max()
                - grouped["observed_success_rate"].min()
            )

            relationship_rows.append({
                "feature": col,
                "analysis": "categorical_target_rate",
                "max_success_rate_difference": diff,
            })

    # Numeric analysis: quantile bins instead of raw unique values
    elif pd.api.types.is_numeric_dtype(s):

        numeric = s.dropna()

        if numeric.nunique() >= 5:

            try:
                bins = pd.qcut(
                    numeric,
                    q=5,
                    duplicates="drop",
                )

                temp = pd.DataFrame({
                    "bin": bins,
                    "target": ml.loc[
                        numeric.index,
                        target_col,
                    ],
                })

                grouped = (
                    temp.groupby(
                        "bin",
                        observed=False,
                    )["target"]
                    .agg(
                        ["size", "mean"]
                    )
                )

                if len(grouped) >= 2:
                    diff = (
                        grouped["mean"].max()
                        - grouped["mean"].min()
                    )

                    relationship_rows.append({
                        "feature": col,
                        "analysis": "numeric_quantile_target_rate",
                        "max_success_rate_difference": diff,
                    })

            except Exception:
                pass

relationship_summary = pd.DataFrame(
    relationship_rows
)

if not relationship_summary.empty:
    relationship_summary = (
        relationship_summary
        .sort_values(
            "max_success_rate_difference",
            ascending=False,
        )
    )

display(relationship_summary)

relationship_summary.to_csv(
    REPORT_DIR / "feature_target_relationships.csv",
    index=False,
)


,feature,analysis,max_success_rate_difference
23,prior_recovered_cases_before,categorical_target_rate,0.810119
9,failure_class_at_decision,categorical_target_rate,0.648903
10,first_failure_class,categorical_target_rate,0.629194
14,instrument_fixed,categorical_target_rate,0.531257
26,hour_of_day,numeric_quantile_target_rate,0.284038
4,amount_inr,categorical_target_rate,0.275718
11,retries_used_before,categorical_target_rate,0.262772
2,segment,categorical_target_rate,0.198736
5,value_tier,categorical_target_rate,0.181777
31,channel,categorical_target_rate,0.168981


## 13. High-Value Intervention Interactions

In [27]:

interaction_tables = []

interaction_specs = [
    ["failure_class", "action_type"],
    ["failure_class_at_decision", "action_type"],
    ["rail", "action_type"],
    ["logging_policy", "action_type"],
    ["value_tier", "action_type"],
]

for cols in interaction_specs:

    if not all(col in ml.columns for col in cols):
        continue

    result = (
        ml.groupby(cols, dropna=False)[target_col]
        .agg(
            observations="size",
            observed_success_rate="mean",
        )
        .reset_index()
    )

    # Add support threshold flag for later policy analysis.
    result["low_support_lt_100"] = (
        result["observations"] < 100
    )

    interaction_tables.append(result)

    print(" × ".join(cols))
    display(
        result.sort_values(
            "observed_success_rate",
            ascending=False,
        )
    )

if interaction_tables:
    pd.concat(
        interaction_tables,
        ignore_index=True,
    ).to_csv(
        REPORT_DIR / "intervention_interactions.csv",
        index=False,
    )


failure_class_at_decision × action_type


,failure_class_at_decision,action_type,observations,observed_success_rate,low_support_lt_100
0,BANK_TECHNICAL,RETRY,13901,0.883390,False
1,BANK_TECHNICAL,SEND_PAYMENT_LINK,2295,0.446187,False
2,BANK_TECHNICAL,SEND_REMINDER,2279,0.423870,False
17,RATE_LIMIT,RETRY,2494,0.389735,False
13,MANDATE_CLOSED,SEND_PAYMENT_LINK,11672,0.270391,False
14,MANDATE_CLOSED,SEND_REMINDER,1964,0.256619,False
12,MANDATE_CLOSED,REQUEST_PM_UPDATE,1946,0.245632,False
5,CARD_EXPIRED,SEND_REMINDER,87,0.229885,True
3,CARD_EXPIRED,REQUEST_PM_UPDATE,288,0.229167,False
8,CARD_LOST_STOLEN,SEND_REMINDER,44,0.227273,True


rail × action_type


,rail,action_type,observations,observed_success_rate,low_support_lt_100
1,CARD_SI,RETRY,25196,0.330648,False
4,UPI_AUTOPAY,REQUEST_PM_UPDATE,1946,0.245632,False
5,UPI_AUTOPAY,RETRY,76318,0.245420,False
0,CARD_SI,REQUEST_PM_UPDATE,430,0.225581,False
6,UPI_AUTOPAY,SEND_PAYMENT_LINK,26913,0.182031,False
2,CARD_SI,SEND_PAYMENT_LINK,3971,0.139008,False
3,CARD_SI,SEND_REMINDER,3501,0.125678,False
7,UPI_AUTOPAY,SEND_REMINDER,19148,0.124608,False


logging_policy × action_type


,logging_policy,action_type,observations,observed_success_rate,low_support_lt_100
3,B2_DOCTRINE,RETRY,30849,0.345684,False
7,B3_RANDOM_LEGAL,RETRY,27874,0.302935,False
4,B2_DOCTRINE,SEND_PAYMENT_LINK,6314,0.256573,False
6,B3_RANDOM_LEGAL,REQUEST_PM_UPDATE,2117,0.243269,False
2,B2_DOCTRINE,REQUEST_PM_UPDATE,259,0.231660,False
1,B1_PSP_PROXY,SEND_PAYMENT_LINK,7124,0.226839,False
0,B1_PSP_PROXY,RETRY,42791,0.185857,False
8,B3_RANDOM_LEGAL,SEND_PAYMENT_LINK,17446,0.126963,False
9,B3_RANDOM_LEGAL,SEND_REMINDER,17489,0.125107,False
5,B2_DOCTRINE,SEND_REMINDER,5160,0.123643,False


value_tier × action_type


,value_tier,action_type,observations,observed_success_rate,low_support_lt_100
5,LOW,RETRY,49747,0.369570,False
4,LOW,REQUEST_PM_UPDATE,1826,0.246988,False
0,HIGH,REQUEST_PM_UPDATE,108,0.231481,False
8,MID,REQUEST_PM_UPDATE,442,0.223982,False
6,LOW,SEND_PAYMENT_LINK,19816,0.207560,False
9,MID,RETRY,37291,0.183690,False
7,LOW,SEND_REMINDER,13558,0.152235,False
10,MID,SEND_PAYMENT_LINK,8455,0.133176,False
1,HIGH,RETRY,14476,0.126140,False
11,MID,SEND_REMINDER,6843,0.092649,False


## 14. Timing / Salary-Proxy Analysis

In [28]:

timing_columns = [
    "proxy_days_to_salary",
    "in_salary_cluster",
    "salary_day_proxy_dom",
    "time_since_first_failure_days",
    "tenure_months",
]

for col in timing_columns:

    if col not in ml.columns:
        continue

    print(f"\n### {col}")

    if pd.api.types.is_numeric_dtype(ml[col]):

        series = pd.to_numeric(
            ml[col],
            errors="coerce",
        )

        if series.nunique(dropna=True) > 10:

            temp = ml[
                [col, target_col]
            ].dropna().copy()

            try:
                temp["bin"] = pd.qcut(
                    temp[col],
                    q=8,
                    duplicates="drop",
                )

                display(
                    temp.groupby(
                        "bin",
                        observed=False,
                    )[target_col]
                    .agg(
                        observations="size",
                        observed_success_rate="mean",
                    )
                )
            except Exception:
                display(
                    series.describe()
                    .to_frame()
                )

        else:
            display(
                ml.groupby(
                    col,
                    dropna=False,
                )[target_col]
                .agg(
                    observations="size",
                    observed_success_rate="mean",
                )
            )

    else:
        display(
            ml.groupby(
                col,
                dropna=False,
            )[target_col]
            .agg(
                observations="size",
                observed_success_rate="mean",
            )
        )



### in_salary_cluster


,observations,observed_success_rate
in_salary_cluster,,
False,96140,0.199688
True,61283,0.272751



### salary_day_proxy_dom


,observations,observed_success_rate
bin,,
"(0.999, 2.0]",24010,0.217451
"(2.0, 4.0]",15586,0.231361
"(4.0, 7.0]",21769,0.222978
"(7.0, 13.0]",19460,0.226465
"(13.0, 24.0]",20671,0.240627
"(24.0, 27.0]",16674,0.240674
"(27.0, 28.0]",39253,0.225155


## 15. Action Support Analysis

In [29]:

support_rows = []

if "action_type" in ml.columns:

    overall = (
        ml["action_type"]
        .value_counts(dropna=False)
        .rename("support_count")
        .reset_index()
    )
    overall.columns = [
        "action_type",
        "support_count",
    ]
    overall["dimension"] = "overall"
    overall["group"] = "all"

    support_rows.append(overall)

    for dim in [
        "failure_class",
        "failure_class_at_decision",
        "rail",
        "value_tier",
        "logging_policy",
    ]:

        if dim not in ml.columns:
            continue

        grouped = (
            ml.groupby(
                [dim, "action_type"],
                dropna=False,
            )
            .size()
            .reset_index(
                name="support_count"
            )
        )

        grouped["dimension"] = dim
        grouped = grouped.rename(
            columns={dim: "group"}
        )

        support_rows.append(
            grouped[
                [
                    "dimension",
                    "group",
                    "action_type",
                    "support_count",
                ]
            ]
        )

if support_rows:
    action_support = pd.concat(
        support_rows,
        ignore_index=True,
    )

    action_support["low_support_lt_100"] = (
        action_support["support_count"] < 100
    )

    display(
        action_support
        .sort_values(
            "support_count"
        )
        .head(100)
    )

    action_support.to_csv(
        REPORT_DIR / "action_support.csv",
        index=False,
    )


,action_type,support_count,dimension,group,low_support_lt_100
12,SEND_REMINDER,44,failure_class_at_decision,CARD_LOST_STOLEN,True
9,SEND_REMINDER,87,failure_class_at_decision,CARD_EXPIRED,True
32,REQUEST_PM_UPDATE,108,value_tier,HIGH,False
23,SEND_REMINDER,110,failure_class_at_decision,RATE_LIMIT,False
22,SEND_PAYMENT_LINK,115,failure_class_at_decision,RATE_LIMIT,False
10,REQUEST_PM_UPDATE,142,failure_class_at_decision,CARD_LOST_STOLEN,False
11,SEND_PAYMENT_LINK,214,failure_class_at_decision,CARD_LOST_STOLEN,False
46,REQUEST_PM_UPDATE,259,logging_policy,B2_DOCTRINE,False
7,REQUEST_PM_UPDATE,288,failure_class_at_decision,CARD_EXPIRED,False
24,REQUEST_PM_UPDATE,430,rail,CARD_SI,False


## 16. Leakage and Decision-Time Review

In [30]:

classification_path = (
    CONTRACT_DIR / "column_classification.csv"
)
availability_path = (
    CONTRACT_DIR / "feature_availability.csv"
)

if classification_path.exists():
    classification = pd.read_csv(
        classification_path
    )
else:
    classification = pd.DataFrame()

if availability_path.exists():
    availability = pd.read_csv(
        availability_path
    )
else:
    availability = pd.DataFrame()

if not classification.empty:
    leakage = classification.copy()

    def leakage_flag(row):
        cls = row["classification"]
        if cls == "TARGET":
            return "DROP - TARGET"
        if cls == "IDENTIFIER":
            return "DROP - IDENTIFIER"
        if cls in {
            "POST-DECISION / POSSIBLE LEAKAGE",
            "OUTCOME / LABEL METADATA",
        }:
            return "DROP - FUTURE/OUTCOME"
        if cls == "UNKNOWN / NEEDS REVIEW":
            return "REVIEW"
        return "CANDIDATE"

    leakage["eda_leakage_flag"] = (
        leakage.apply(leakage_flag, axis=1)
    )

    display(leakage)

    leakage.to_csv(
        REPORT_DIR / "leakage_review.csv",
        index=False,
    )


,column,classification,usable_for_ml,reason,source,eda_leakage_flag
0,logging_policy,POLICY / LOGGING,False,Describes the policy that generated/logged the...,ml_datamart,CANDIDATE
1,case_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart,DROP - IDENTIFIER
2,subscription_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart,DROP - IDENTIFIER
3,customer_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart,DROP - IDENTIFIER
4,decision_seq,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart,CANDIDATE
5,decision_ts_h,TEMPORAL SOURCE FIELD,False,Used to derive decision-time temporal features...,ml_datamart,CANDIDATE
6,failure_class_at_decision,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart,CANDIDATE
7,first_failure_class,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart,CANDIDATE
8,rail,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart,CANDIDATE
9,bank_name,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart,CANDIDATE


## 17. Cross-Table Behavioural Consistency

In [31]:

consistency_rows = []

# Failure class consistency between payment attempts and ML datamart
if (
    "failure_class" in data["payment_attempts"].columns
    and "failure_class" in ml.columns
):
    consistency_rows.append({
        "check": "failure_class_present_in_attempts_and_ml",
        "attempt_classes": data["payment_attempts"]["failure_class"]
            .dropna().nunique(),
        "ml_classes": ml["failure_class"]
            .dropna().nunique(),
    })

# Rail consistency
if "rail" in ml.columns and "rail" in data["subscriptions"].columns:
    ml_rails = set(
        ml["rail"].dropna().astype(str)
    )
    sub_rails = set(
        data["subscriptions"]["rail"].dropna().astype(str)
    )

    consistency_rows.append({
        "check": "ml_rail_values_subset_of_subscription_rail_values",
        "value": ml_rails.issubset(sub_rails),
        "ml_only_values": sorted(
            ml_rails - sub_rails
        ),
    })

display(pd.DataFrame(consistency_rows))


,check,value,ml_only_values
0,ml_rail_values_subset_of_subscription_rail_values,True,[]


## 18. Evidence-Based Feature Decision Gate

**Do not automatically select final features from thresholds here.**

C2 provides evidence for C3.

Classify variables into:
- KEEP candidate
- TRANSFORM candidate
- DROP candidate
- REVIEW

based on actual:
- missingness
- cardinality
- variation
- target relationship
- action support
- leakage status
- semantic availability

Do not call an action "best" because it has a higher observed success rate.
Observed policy/action differences are not automatically causal.

In [32]:

decision_rows = []

for _, row in feature_summary.iterrows():

    feature = row["feature"]
    classification_row = classification[
        classification["column"] == feature
    ] if not classification.empty else pd.DataFrame()

    cls = (
        classification_row.iloc[0]["classification"]
        if not classification_row.empty
        else "UNKNOWN / NEEDS REVIEW"
    )

    if cls == "UNKNOWN / NEEDS REVIEW":
        decision = "REVIEW"
        reason = "Semantics not established in C0."
    elif cls in {
        "TARGET",
        "IDENTIFIER",
        "POLICY / LOGGING",
        "OUTCOME / LABEL METADATA",
        "POST-DECISION / POSSIBLE LEAKAGE",
        "TEMPORAL SOURCE FIELD",
    }:
        decision = "DROP / EXCLUDE FROM X"
        reason = cls
    else:
        missing_pct = row["missing_pct"]
        nunique = row["nunique"]

        if missing_pct > 95:
            decision = "REVIEW/DROP"
            reason = "Extremely high missingness."
        elif nunique <= 1:
            decision = "DROP"
            reason = "Constant feature."
        else:
            decision = "CANDIDATE"
            reason = "Requires C3 final selection after full EDA review."

    decision_rows.append({
        "feature": feature,
        "c0_classification": cls,
        "c2_decision": decision,
        "reason": reason,
    })

feature_gate = pd.DataFrame(decision_rows)
display(feature_gate)

feature_gate.to_csv(
    REPORT_DIR / "feature_decision_gate.csv",
    index=False,
)


,feature,c0_classification,c2_decision,reason
0,rail,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
1,bank_name,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
2,segment,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
3,plan_tier,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
4,amount_inr,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
5,value_tier,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
6,subscription_age_days,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
7,sub_billing_day,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
8,decision_seq,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...
9,failure_class_at_decision,OBSERVABLE DECISION-TIME FEATURE,CANDIDATE,Requires C3 final selection after full EDA rev...


## 19. Final EDA Findings

This section intentionally does not contain pre-written numerical findings.

The final `EDA_FINDINGS.md` should be written only after reviewing the exported
tables/plots and recording actual evidence.

Each finding must state:
- observed value/statistic
- dataset/table
- slice/group
- interpretation
- ML implication

Then C3 will finalize transformations and the feature contract.

In [33]:

# Export a factual summary template, without inventing conclusions.
summary = {
    "total_ml_rows": int(len(ml)),
    "unique_cases": (
        int(ml["case_id"].nunique())
        if "case_id" in ml.columns else None
    ),
    "unique_customers": (
        int(ml["customer_id"].nunique())
        if "customer_id" in ml.columns else None
    ),
    "target_column": target_col,
    "target_missing": int(ml[target_col].isna().sum()),
    "target_positive_rate": float(ml[target_col].mean()),
    "candidate_feature_count": int(
        len(candidates)
    ),
}

with open(
    REPORT_DIR / "eda_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(summary, f, indent=2)

print("C2 analysis completed.")
print("Review reports/eda/ before making C3 decisions.")


C2 analysis completed.
Review reports/eda/ before making C3 decisions.
